# 14장. 반복되는 분석 흐름을 안전하게 자동화하기

이 노트북은 `book/chapters/ch14_airflow_pipeline.md`와 `src/automation_pipeline.py`의 현재 구현을 단계별로 확인합니다.

핵심 원칙:
- Airflow 전에 동일 Python 파이프라인을 검증합니다.
- 완료 주문 기준 금액과 회계상 순매출을 구분합니다.
- PK/FK, `line_total`, 집계 총합, Freshness, Run ID를 검증합니다.
- 파일 단위 원자적 교체는 전체 파이프라인 트랜잭션과 같지 않으므로 Run ID로 교차 산출물 일관성을 확인합니다.
- Airflow Task Green과 분석 Validation PASS를 구분합니다.
- Docker/Airflow 명령은 아래에서 설명하지만 이 Notebook 셀이 자동 실행하지 않습니다.

## 0. 실행 전 확인

원본 파일이 없다면 프로젝트 루트에서 먼저 `python scripts/generate_sample_data.py`를 실행합니다.
Docker Compose 환경은 로컬 학습·탐색용이며 운영 배포 템플릿이 아닙니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
REPORT_DIR = PROJECT_ROOT / 'reports'
AIRFLOW_DIR = PROJECT_ROOT / 'automation' / 'airflow'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('프로젝트 루트:', PROJECT_ROOT)
print('Airflow 폴더:', AIRFLOW_DIR)

In [ ]:
from src.automation_pipeline import (
    check_input_files,
    create_airflow_setup_guide,
    create_pipeline_task_summary,
    generate_report,
    generate_visualizations,
    run_analysis,
    run_local_pipeline,
    run_preprocessing,
    validate_outputs,
)

## 1. Task 계약을 먼저 확인합니다

In [ ]:
task_contract = create_pipeline_task_summary()
task_contract

각 Task에는 입력·출력·실패 조건·재시도 정책·멱등성 기준이 있습니다. 입력 누락이나 스키마 오류처럼 결정적인 실패는 무작정 재시도하지 않습니다.

## 2. 로컬 Python 파이프라인을 단계별로 확인합니다

In [ ]:
input_check = check_input_files(PROJECT_ROOT)
input_check

In [ ]:
preprocessing_outputs = run_preprocessing(PROJECT_ROOT)
pd.DataFrame({
    'dataset': preprocessing_outputs.keys(),
    'path': [str(path) for path in preprocessing_outputs.values()],
})

전처리는 `customers`, `products`, `orders`, `order_items`의 PK를 확인하며 `order_items.order_item_id`도 고유해야 합니다. FK와 `line_total = quantity × unit_price`가 맞지 않으면 중단합니다.

In [ ]:
analysis_outputs = run_analysis(PROJECT_ROOT)
daily_sales = pd.read_csv(REPORT_DIR / 'ch14_daily_sales.csv')
category_sales = pd.read_csv(REPORT_DIR / 'ch14_category_sales.csv')
run_metadata = pd.read_csv(REPORT_DIR / 'ch14_pipeline_run_metadata.csv')
display(daily_sales.head())
display(category_sales.head())
display(run_metadata)

금액 범위는 `order_status == completed`인 주문 상세의 `quantity × unit_price` 합계입니다. `daily_sales`, `category_sales`, `run_metadata`는 동일한 `pipeline_run_id`를 가져야 합니다.

In [ ]:
figure_outputs = generate_visualizations(PROJECT_ROOT)
report_path = generate_report(PROJECT_ROOT)
print(figure_outputs)
print(report_path)

In [ ]:
validation_log = validate_outputs(PROJECT_ROOT)
validation_log

Validation은 파일 존재뿐 아니라 수정 시각, CSV 행 수, Run ID, daily/category 총합, 카테고리 비율 합, 보고서의 completed Scope를 확인합니다. 하나라도 `error`이면 외부 전달 단계로 넘어가지 않습니다.

## 3. 전체 로컬 파이프라인을 한 번에 재실행합니다

In [ ]:
pipeline_result = run_local_pipeline(PROJECT_ROOT)
print('Pipeline Run ID:', pipeline_result['pipeline_run_id'])
pipeline_result['validation_log']

터미널에서는 `python scripts/run_ch14_pipeline.py`로 같은 흐름을 실행합니다. 같은 입력을 다시 실행해도 CSV가 append되지 않고 각 파일을 전체 교체합니다.

## 4. Docker Compose 학습 환경을 정적으로 점검합니다

In [ ]:
compose_files = [
    AIRFLOW_DIR / 'Dockerfile',
    AIRFLOW_DIR / 'docker-compose.yml',
    AIRFLOW_DIR / 'requirements.txt',
    AIRFLOW_DIR / '.env.example',
    AIRFLOW_DIR / 'dags' / 'ch14_local_analysis_pipeline.py',
]
pd.DataFrame([
    {
        'file': str(path.relative_to(PROJECT_ROOT)),
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() else 0,
    }
    for path in compose_files
] )

`.env.example`은 실제 비밀값이 아니라 빈 Secret 슬롯만 가져야 합니다. 실제 `.env` 값은 Notebook에서 읽거나 출력하지 않습니다.

In [ ]:
env_example_text = (AIRFLOW_DIR / '.env.example').read_text(encoding='utf-8')
env_lines = {
    line.split('=', 1)[0]: line.split('=', 1)[1]
    for line in env_example_text.splitlines()
    if line and not line.startswith('#') and '=' in line
}
secret_names = [
    'AIRFLOW_DB_PASSWORD',
    'AIRFLOW_API_JWT_SECRET',
    '_AIRFLOW_WWW_USER_PASSWORD',
]
pd.DataFrame({
    'name': secret_names,
    'declared': [name in env_lines for name in secret_names],
    'example_value_empty': [env_lines.get(name, None) == '' for name in secret_names],
})

## 5. Canonical DAG 설정을 정적으로 확인합니다

실제 DAG는 `automation/airflow/dags/ch14_local_analysis_pipeline.py` 하나를 기준으로 합니다. 루트 `dags/`의 옛 파일은 legacy 안내만 남아 있습니다.

In [ ]:
dag_text = (AIRFLOW_DIR / 'dags' / 'ch14_local_analysis_pipeline.py').read_text(encoding='utf-8')
dag_checks = {
    'Airflow 3 public SDK': 'from airflow.sdk import dag, task' in dag_text,
    'timezone start_date': 'pendulum.datetime' in dag_text and 'tz=DAG_TIMEZONE' in dag_text,
    'manual schedule': 'schedule=None' in dag_text,
    'catchup disabled': 'catchup=False' in dag_text,
    'one active DAG run': 'max_active_runs=1' in dag_text,
    'bounded concurrency': 'max_active_tasks=2' in dag_text,
    'execution timeout': 'execution_timeout' in dag_text,
    'visual/report branch': 'analysis_task >> [visualization_task, report_task]' in dag_text,
    'validation after both': '[visualization_task, report_task] >> validation_task' in dag_text,
}
pd.Series(dag_checks, name='configured')

## 6. Docker/Airflow 실행 순서 — Notebook이 자동 실행하지 않는 명령

```bash
cd automation/airflow
cp .env.example .env
# Windows PowerShell: Copy-Item .env.example .env
# 실제 .env에서 서로 다른 Secret 3개를 채운 뒤
docker compose build
docker compose up airflow-init
docker compose up -d
docker compose ps
```

수업 실습에서만 환경을 확인한 뒤 직접 실행합니다. `airflow-init`은 종료 코드 0, 필수 서비스는 running/healthy인지 확인합니다.

In [ ]:
create_airflow_setup_guide()

## 7. 안전한 실패·복구 실습

원본을 삭제하지 말고 **샘플 데이터에서만** `customers.csv`를 `customers.csv.exercise-backup`으로 임시 이름 변경합니다. `check_input_files`가 실패하고 누락 경로를 로그에 남기는지 확인한 뒤 즉시 원래 이름으로 복구합니다. 실제 업무 데이터에서는 수행하지 않습니다.

## 8. 문제 진단 원칙

- 재시작 반복: 메모리와 서비스 로그 확인
- 8080 충돌: `.env`의 `AIRFLOW_UI_PORT` 변경
- ModuleNotFoundError: requirements와 image build log 확인
- Linux 권한: `AIRFLOW_UID` 확인, 무조건 `chmod -R 777` 금지
- JWT/API 인증: 실제 Secret을 출력하지 말고 모든 서비스가 같은 Compose 변수에서 받는지 확인
- Stale Result: validation의 Freshness와 Run ID 확인

## 9. 종료와 파괴적 초기화를 구분합니다

일반 종료: `docker compose down`

파괴적 초기화: `docker compose down --volumes --remove-orphans`

두 번째 명령은 Postgres volume과 Airflow 실행 기록·계정 정보를 지울 수 있으므로 완전 초기화가 필요한 학습 환경에서만 사용합니다.

## 10. 외부 전달 Gate

Make/n8n/Gmail/Slack/Drive 전달은 `validate_outputs`가 PASS한 뒤에만 설계합니다. Retry나 재실행으로 같은 보고서를 중복 전송하지 않도록 Airflow Dag Run ID 또는 별도의 delivery idempotency key를 기록해야 합니다. 개인정보·주문 식별자가 든 내부 파일은 공개 채널로 보내지 않습니다.

## 11. 최종 확인

좋은 자동화는 `Task Green`을 만드는 것이 아니라 **같은 입력을 안전하게 재실행하고, 최신의 같은 Run 산출물인지 검증하고, FAIL이면 다음 단계와 외부 전달을 막는 것**입니다.